### Bare Bones Animation using Pygame

First we need import pygame

In [1]:
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.13.2)
Hello from the pygame community. https://www.pygame.org/contribute.html


Then we need to initialize the module

In [2]:
pygame.init()

(5, 0)

Then create a Clock object that can be used to track an amount of time. The clock also provides several functions to help control a game's framerate. 


In [3]:
clock = pygame.time.Clock()

Set the window size to 400 by 300 and get the screen object

In [4]:
screen = pygame.display.set_mode((400,300))

### Animation Loop
In order to display an animation, you need to have a loop. The loop while first clear the screen then draw whatever elements is needed then the loop is repeated. You have something like this:


    while true
       clear the screen
       draw items

   
This will loop forever, so we want the loop to monitor mouse events so that the loop can terminate. Also the loop may be going to fast that nothing can be seen. So we often put a timer within the loop. This done using the Clock.tick() function. Clock.tick(x) will make pygame to render only x times per second. This function is used in the rendering will block execution until 1/60 seconds (if x is 60) have passed since the previous time clock.tick was called in the loop.

        


### Getting Events from Pygame

We can get the events using: 

    for event in pygame.event.get():

We check that the events is a quit event:

    for event in pygame.event.get():
      if event.type == pygame.QUIT:
           done = True
           pygame.quit()

### Drawing Items in the Loop

In order to draw items, we need to clear the screen and draw on it repeatedly.

We clear the screen with:

    screen.fill((0, 0, 0))

We can draw a simple object with:

    pygame.draw.rect(screen, color, pygame.Rect(loc_x,loc_y, width, height))

We can draw random rectangles if we set loc_x and loc_y with random integer values


    loc_x=random.randint(0,w)
    loc_y=random.randint(0,h)
    pygame.draw.rect(screen, color, pygame.Rect(loc_x, loc_y, rw,rh)) #draw rectangle


### Double Buffering

If you are drawing something on the screen while displaying, the animation will not be smooth. Therefore many libraries use a double buffering system. This means there are two memory locations for storing display objects. One of the buffer is used for display. While one of the buffer is displayed, the other buffer is used for drawing and update objects. Then the buffers are swapped, what was previously drawn will now be displayed, and the other buffer will be used for drawing. The swapping continues until the animation loop is terminated. 

The buffer flipping is done with:

    pygame.display.flip()

### We put all of these together 

In [6]:
import pygame
import random

pygame.init()
clock = pygame.time.Clock()

#initialize values
loc_x=0
loc_y=30
rw,rh=(30,30)
color=(255, 128, 0)
w,h=(400,300)

screen = pygame.display.set_mode((w,h))
done = False

while not done:
        for event in pygame.event.get():
                if event.type == pygame.QUIT:
                        done = True
                        
        if done:
            break
        screen.fill((0, 0, 0)) # clear the screen
        loc_x=random.randint(0,w)
        loc_y=random.randint(0,h)
        pygame.draw.rect(screen, color, pygame.Rect(loc_x, loc_y, rw,rh)) #draw rectangle
        pygame.display.flip()
        
        clock.tick(100)
            
pygame.quit()       

### Let's add Key Press

Remove the random location code. Now we want to control the square by the arrow keys. So we need to detect the key press events. Add these lines after the event.get() line.



    pressed = pygame.key.get_pressed()
    if pressed[pygame.K_UP]: loc_y -= 3
    if pressed[pygame.K_DOWN]: loc_y += 3
    if pressed[pygame.K_LEFT]: loc_x -= 3
    if pressed[pygame.K_RIGHT]: loc_x += 3


To run the code below, you need to give focus to the pygame window - that mean's click on it first, then only it can detect the keypresses.

In [8]:
import pygame
import random

pygame.init()
clock = pygame.time.Clock()

#initialize values
loc_x=0
loc_y=30
rw,rh=(30,30)
color=(255, 128, 0)
w,h=(400,300)

screen = pygame.display.set_mode((w,h))
done = False


    

while not done:
        for event in pygame.event.get():
                if event.type == pygame.QUIT:
                        done = True
                        
        if done:
            break
        screen.fill((0, 0, 0)) # clear the screen
        pressed = pygame.key.get_pressed()
        if pressed[pygame.K_UP]: loc_y -= 3
        if pressed[pygame.K_DOWN]: loc_y += 3
        if pressed[pygame.K_LEFT]: loc_x -= 3
        if pressed[pygame.K_RIGHT]: loc_x += 3
        pygame.draw.rect(screen, color, pygame.Rect(loc_x, loc_y, rw,rh)) #draw rectangle
        pygame.display.flip()
        
        clock.tick(100)
            
pygame.quit()       

### Let's a have Ball

Let's convert the original square into a paddle. Then we add ball that is moving on its own. The objective is to use the paddle to hit the ball so that it does not fall through the lower boundary. 

You can create a ball that moves on its own like this:

        #ball
        ballx+=vx
        bally+=vy
        pygame.draw.rect(screen, bcolor, pygame.Rect(ballx, bally, bw,bh)) #draw ball

In [5]:
import pygame
import random

pygame.init()
clock = pygame.time.Clock()

#initialize values
loc_x=150
loc_y=200
rw,rh=(30,30)
color=(255, 128, 0)
w,h=(400,300)
bcolor = (0, 255, 255)
ballx = 200
bally = 150
bw, bh = (20, 20)
vy = 3
vx = 3

screen = pygame.display.set_mode((w,h))
done = False

def get_keypress(x, y):
    pressed = pygame.key.get_pressed()
    if pressed[pygame.K_UP]: y -= 3
    if pressed[pygame.K_DOWN]: y += 3
    if pressed[pygame.K_LEFT]: x -= 3
    if pressed[pygame.K_RIGHT]: x += 3
    return x, y

while not done:
        for event in pygame.event.get():
                if event.type == pygame.QUIT:
                        done = True
                        
        if done:
            break
        screen.fill((0, 0, 0)) # clear the screen

        loc_x, loc_y = get_keypress(loc_x, loc_y)

        ballx += vx
        bally += vy

        pygame.draw.rect(screen, color, pygame.Rect(loc_x, loc_y, rw,rh)) #draw rectangle
        pygame.draw.rect(screen, bcolor, pygame.Rect(ballx, bally, bw,bh)) #draw ball

        pygame.display.flip()
        
        clock.tick(100)
            
pygame.quit()       

What is the problem with the above code? It is not able to detect when it goes out of bounds, and can't detect that it is hitting the paddle. We need to add collision detection code!!! How do we do that?

Test out of bounds:

        #bounce on wall
        if ballx>w-bw or ballx<0:
            vx=vx*-1
        if bally>h-bh or bally<0:
            vy=vy*-1


Test collision with Paddle:

        #collision detection
        if pygame.Rect(loc_x,loc_y,rw,rh).colliderect(pygame.Rect(ballx,bally,bw,bh)):
               vy=vy*-1

In [6]:
import pygame


def bounce_wall(x, y, vx, vy, w, h, bw, bh):
    if x <= 0 or x + bw >= w:
        vx = -vx
    if y <= 0 or y + bh >= h:
        vy = -vy
    return vx, vy

def detect_collision(rect_x, rect_y, rect_w, rect_h, ball_x, ball_y, ball_w, ball_h):
    if pygame.Rect(rect_x,rect_y,rect_w,rect_h).colliderect(pygame.Rect(ball_x,ball_y,ball_w,ball_h)):
        return True
    return False

pygame.init()
clock = pygame.time.Clock()

#initialize values
loc_x=150
loc_y=200
rw,rh=(30,30)
color=(255, 128, 0)
w,h=(400,300)
bcolor = (0, 255, 255)
ballx = 200
bally = 150
bw, bh = (20, 20)
vy = 3
vx = 3

screen = pygame.display.set_mode((w,h))
done = False

def get_keypress(x, y):
    pressed = pygame.key.get_pressed()
    if pressed[pygame.K_UP]: y -= 3
    if pressed[pygame.K_DOWN]: y += 3
    if pressed[pygame.K_LEFT]: x -= 3
    if pressed[pygame.K_RIGHT]: x += 3
    return x, y

while not done:
        for event in pygame.event.get():
                if event.type == pygame.QUIT:
                        done = True
                        
        if done:
            break
        screen.fill((0, 0, 0)) # clear the screen

        loc_x, loc_y = get_keypress(loc_x, loc_y)

        if detect_collision(loc_x, loc_y, rw, rh, ballx, bally, bw, bh):
            vx = -vx
            vy = -vy
        
        vx, vy = bounce_wall(ballx, bally, vx, vy, w, h, bw, bh)
        ballx += vx
        bally += vy

        pygame.draw.rect(screen, color, pygame.Rect(loc_x, loc_y, rw,rh)) #draw rectangle
        pygame.draw.rect(screen, bcolor, pygame.Rect(ballx, bally, bw,bh)) #draw ball

        pygame.display.flip()
        
        clock.tick(100)
            
pygame.quit()       

### Additional Exercise (optional)

1. Modify so that the ball will start at a random position.
1. Modify so that the ball will drop through the lower boundary (floor) if the paddle did not hit it.
1. Add a score counting for number of missed ball (those that fall through the floor) and display the score on screen.
1. Add another player. The second player control the paddle on the top. 
1. Change the orientation so that two players play by controlling the paddle on the left and right. 

Code and Worksheet by Loke K.S.